In [1]:
from google.colab import files
uploaded = files.upload()

Saving away_team.csv to away_team.csv
Saving home_team.csv to home_team.csv


In [2]:
from google.colab import files
uploaded = files.upload()

Saving venue.csv to venue.csv
Saving event.csv to event.csv
Saving statistics.csv to statistics.csv


In [3]:
import pandas as pd

files = {
    'event': 'event.csv',
    'home_team': 'home_team.csv',
    'away_team': 'away_team.csv',
    'venue': 'venue.csv',
    'statistics': 'statistics.csv',
}
data = {name: pd.read_csv(path) for name, path in files.items()}

for name, df in data.items():
    print(f"{name}: {df.shape}")

def dedupe_one_per_match(df, id_col='match_id'):
    df = df.drop_duplicates()
    df = df.assign(_null_count=df.isnull().sum(axis=1))
    df = df.sort_values('_null_count').drop_duplicates(subset=id_col, keep='first')
    df = df.drop(columns='_null_count').reset_index(drop=True)
    return df

one_row_per_match = ['event', 'home_team', 'away_team', 'venue']
for name in one_row_per_match:
    before = len(data[name])
    data[name] = dedupe_one_per_match(data[name])
    print(f"{name}: {before} -> {len(data[name])}")

before = len(data['statistics'])
data['statistics'] = data['statistics'].drop_duplicates().reset_index(drop=True)
print(f"statistics: {before} -> {len(data['statistics'])}")

stat_dup_check = data['statistics'].duplicated(subset=['match_id', 'statistic_name', 'period']).sum()
print(f"statistics remaining duplicate (match_id, statistic_name, period) combos: {stat_dup_check}")

if stat_dup_check > 0:
    data['statistics'] = data['statistics'].assign(_null_count=data['statistics'].isnull().sum(axis=1))
    data['statistics'] = data['statistics'].sort_values('_null_count').drop_duplicates(
        subset=['match_id', 'statistic_name', 'period'], keep='first'
    ).drop(columns='_null_count').reset_index(drop=True)
    print(f"statistics after resolving duplicates: {len(data['statistics'])}")

data['event']['start_datetime'] = pd.to_datetime(data['event']['start_datetime'], unit='s')

for t in ['home_team', 'away_team']:
    data[t]['current_rank'] = pd.to_numeric(data[t]['current_rank'], errors='coerce')

data['statistics']['home_value'] = pd.to_numeric(data['statistics']['home_value'], errors='coerce')
data['statistics']['away_value'] = pd.to_numeric(data['statistics']['away_value'], errors='coerce')

print("\nUnique statistic_category_name values:")
print(data['statistics']['statistic_category_name'].unique())
print("\nUnique statistic_name values:")
print(data['statistics']['statistic_name'].unique())

event: (35053, 10)
home_team: (25610, 18)
away_team: (24203, 18)
venue: (35423, 5)
statistics: (1358234, 13)
event: 35053 -> 16873
home_team: 25610 -> 12389
away_team: 24203 -> 11690
venue: 35423 -> 16749
statistics: 1358234 -> 746361
statistics remaining duplicate (match_id, statistic_name, period) combos: 80759
statistics after resolving duplicates: 665602

Unique statistic_category_name values:
['service' 'return' 'games' 'points' 'miscellaneous']

Unique statistic_name values:
['second_serve_points' 'first_serve_points' 'second_serve' 'first_serve'
 'second_serve_return_points' 'first_serve_return_points'
 'break_points_saved' 'max_games_in_a_row' 'max_points_in_a_row'
 'receiver_points_won' 'service_points_won' 'service_games_played'
 'double_faults' 'aces' 'tiebreaks' 'break_points_converted'
 'return_games_played' 'service_games_won' 'total_won' 'total']


**Question15**

In [6]:
import pandas as pd

# ---------- gam 1: keshvare bazikonan (home + away) be soorate set ----------
home_countries_set = set(data['home_team']['country'].dropna().unique())
away_countries_set = set(data['away_team']['country'].dropna().unique())
player_countries_set = home_countries_set | away_countries_set
print(f"Distinct player countries: {len(player_countries_set)}")

# ---------- gam 2: keshvare venue (mahale bargozari) ----------
venue_countries_set = set(data['venue']['country'].dropna().unique())
print(f"Distinct venue countries: {len(venue_countries_set)}")

# ---------- gam 3: keshvarhaii ke faghat too venue hastan na too player ----------
venue_only = venue_countries_set - player_countries_set
print(f"Countries only in venue (host but no player from there): {len(venue_only)}")
print(sorted(venue_only))

# ---------- gam 4: ejtemae (union) har do majmoo'e ----------
all_countries_union = player_countries_set | venue_countries_set
print(f"\nTotal distinct countries (players UNION venues): {len(all_countries_union)}")

# ---------- gam 5: OVERALL ANSWER ----------
print("\n" + "="*75)
print("OVERALL ANSWER")
print("="*75)
print(f"Distinct countries (players only)         : {len(player_countries_set)}")
print(f"Distinct countries (tournament venue only): {len(venue_countries_set)}")
print(f"Distinct countries (union of both)         : {len(all_countries_union)}")

Distinct player countries: 101
Distinct venue countries: 64
Countries only in venue (host but no player from there): 7
['Bahrain', 'England', 'Rwanda', 'Scotland', 'Sri Lanka', 'Togo', 'United Arab Emirates']

Total distinct countries (players UNION venues): 108

OVERALL ANSWER
Distinct countries (players only)         : 101
Distinct countries (tournament venue only): 64
Distinct countries (union of both)         : 108


**Question16**

In [7]:
import pandas as pd

# ---------- gam 1: amadesazi event (winner_code) ----------
matches = data['event'][['match_id', 'winner_code']].dropna(subset=['winner_code'])
matches['winner_code'] = matches['winner_code'].astype(int)
matches = matches[matches['winner_code'].isin([1, 2])]
print(f"Valid matches with winner: {len(matches)}")

# ---------- gam 2: etesal be home/away baraye player_id va rank ----------
h = data['home_team'][['match_id', 'player_id', 'name', 'current_rank']].rename(
    columns={'player_id': 'home_pid', 'name': 'home_name', 'current_rank': 'home_rank'})
a = data['away_team'][['match_id', 'player_id', 'name', 'current_rank']].rename(
    columns={'player_id': 'away_pid', 'name': 'away_name', 'current_rank': 'away_rank'})

matches = matches.merge(h, on='match_id', how='inner').merge(a, on='match_id', how='inner')
matches = matches.dropna(subset=['home_pid', 'away_pid'])
print(f"Matches with both player IDs known: {len(matches)}")

# ---------- gam 3: tabdile har match be do radif (didgahe har bazikon) ----------
p1 = matches[['match_id', 'home_pid', 'home_name', 'away_rank', 'winner_code']].copy()
p1['is_win'] = (p1['winner_code'] == 1).astype(int)
p1 = p1.rename(columns={'home_pid': 'player_id', 'home_name': 'player_name', 'away_rank': 'opponent_rank'})

p2 = matches[['match_id', 'away_pid', 'away_name', 'home_rank', 'winner_code']].copy()
p2['is_win'] = (p2['winner_code'] == 2).astype(int)
p2 = p2.rename(columns={'away_pid': 'player_id', 'away_name': 'player_name', 'home_rank': 'opponent_rank'})

long = pd.concat([
    p1[['match_id', 'player_id', 'player_name', 'opponent_rank', 'is_win']],
    p2[['match_id', 'player_id', 'player_name', 'opponent_rank', 'is_win']]
], ignore_index=True)
print(f"Total player-match rows: {len(long)}")

# ---------- gam 4: filter faghat baziha moghabele hArife top10 ----------
vs_top10 = long.dropna(subset=['opponent_rank'])
vs_top10 = vs_top10[vs_top10['opponent_rank'] <= 10]
print(f"Player-match rows vs top-10 opponent: {len(vs_top10)}")

# ---------- gam 5: groupbandi va mohasebe darsade bord ----------
summary = vs_top10.groupby(['player_id', 'player_name']).agg(
    matches=('is_win', 'count'), wins=('is_win', 'sum')
).reset_index()
summary['win_pct'] = summary['wins'] / summary['matches'] * 100

# ---------- gam 6: hazfe bazikonanike faghat 1-2 baazi dashtan (natije gomrah konande) ----------
MIN_MATCHES = 5
summary_reliable = summary[summary['matches'] >= MIN_MATCHES].copy()
print(f"\nPlayers with >= {MIN_MATCHES} matches vs top-10: {len(summary_reliable)}")
print(f"Players excluded (too few matches, e.g. 100% from 1 match): {len(summary) - len(summary_reliable)}")

summary_reliable = summary_reliable.sort_values('win_pct', ascending=False)
print("\nTop 10 (reliable, min 5 matches):")
print(summary_reliable.head(10))

# ---------- gam 7: OVERALL ANSWER ----------
top_player = summary_reliable.iloc[0]

print("\n" + "="*75)
print("OVERALL ANSWER")
print("="*75)
print(f"Note: based on CURRENT rank of opponents, not rank at match time.")
print(f"Player            : {top_player['player_name']} (player_id={int(top_player['player_id'])})")
print(f"Matches vs top-10  : {int(top_player['matches'])}")
print(f"Wins               : {int(top_player['wins'])}")
print(f"Win percentage     : {top_player['win_pct']:.1f}%")

Valid matches with winner: 16266
Matches with both player IDs known: 9608
Total player-match rows: 19216
Player-match rows vs top-10 opponent: 234

Players with >= 5 matches vs top-10: 4
Players excluded (too few matches, e.g. 100% from 1 match): 131

Top 10 (reliable, min 5 matches):
    player_id    player_name  matches  wins  win_pct
84     179146  Kalinskaya A.        5     4     80.0
12      23581    Dimitrov G.        5     3     60.0
97     201239   de Minaur A.        5     3     60.0
6       19017    Azarenka V.        5     2     40.0

OVERALL ANSWER
Note: based on CURRENT rank of opponents, not rank at match time.
Player            : Kalinskaya A. (player_id=179146)
Matches vs top-10  : 5
Wins               : 4
Win percentage     : 80.0%


**Question17**

In [8]:
import pandas as pd

# ---------- gam 1: filter faghat radifhaye break_points_converted ba period == ALL ----------
bp_all = data['statistics'][
    (data['statistics']['statistic_name'] == 'break_points_converted') &
    (data['statistics']['period'] == 'ALL')
].copy()
print(f"Match rows with ALL-period break data: {len(bp_all)}")

# ---------- gam 2: barresi tekrari boodane match_id (bayad yekta bashe) ----------
dup = bp_all['match_id'].duplicated().sum()
print(f"Duplicate match_id in break_points_converted (ALL): {dup}")
if dup > 0:
    bp_all = bp_all.drop_duplicates(subset='match_id', keep='first')

# ---------- gam 3: barresie meghdare gomshode ya manfi ----------
missing = bp_all[['home_value', 'away_value']].isnull().any(axis=1).sum()
print(f"Rows with missing home/away value: {missing}")
bp_all = bp_all.dropna(subset=['home_value', 'away_value'])

negative = bp_all[(bp_all['home_value'] < 0) | (bp_all['away_value'] < 0)]
print(f"Rows with negative break values: {len(negative)}")
bp_all = bp_all[(bp_all['home_value'] >= 0) & (bp_all['away_value'] >= 0)]

# ---------- gam 4: mohasebe tedade kole break dar har mosabeghe ----------
bp_all['total_breaks'] = bp_all['home_value'] + bp_all['away_value']

# ---------- gam 5: barresie meghdare part (yek mosabeghe ma'mooli 0 ta 15 break dare) ----------
print("\nTotal breaks per match describe:")
print(bp_all['total_breaks'].describe())

outliers = bp_all[bp_all['total_breaks'] > 20]
print(f"Suspicious matches with >20 breaks: {len(outliers)}")
bp_clean = bp_all[bp_all['total_breaks'] <= 20]

# ---------- gam 6: OVERALL ANSWER ----------
avg_breaks = bp_clean['total_breaks'].mean()

print("\n" + "="*75)
print("OVERALL ANSWER")
print("="*75)
print(f"Matches analyzed         : {len(bp_clean)}")
print(f"Average breaks per match : {avg_breaks:.2f}")

Match rows with ALL-period break data: 11393
Duplicate match_id in break_points_converted (ALL): 0
Rows with missing home/away value: 0
Rows with negative break values: 0

Total breaks per match describe:
count    11393.000000
mean         6.504959
std          3.292878
min          0.000000
25%          4.000000
50%          6.000000
75%          8.000000
max         29.000000
Name: total_breaks, dtype: float64
Suspicious matches with >20 breaks: 8

OVERALL ANSWER
Matches analyzed         : 11385
Average breaks per match : 6.49
